# MARL Cops & Robbers — §9 analysis (SDK-only)

This notebook consumes **only** the `MarlSDK` (no direct `env` / `marl` / `mcp` / learner imports).

## Formalism — Dec-POMDP / POSG

The team faces a Dec-POMDP $\langle \mathcal{I}, S, \{A_i\}, T, R, \{\Omega_i\}, O, \gamma \rangle$ — agents $i\in\mathcal{I}$ share one team reward $R$ and each sees only a local observation $o_i\in\Omega_i$ via $O(o_i\mid s', a)$ (eq 1). Because cop and thief have *opposing* objectives the true game is a **POSG** (eq 3); we collapse the adversary into the environment via alternating best-response self-play (a deliberate Dec-POMDP proxy — named as a limitation in README §7).

## Value decomposition

**IGM** (eq 5): $\arg\max_{\mathbf{a}} Q_{tot}(s,\mathbf{a}) = \big(\arg\max_{a_i} Q_i(o_i,a_i)\big)_{i\in\mathcal{I}}$.

**VDN** (eq 6, additive special case): $Q_{tot}(s,\mathbf{a}) = \sum_{i} Q_i(o_i, a_i)$.

**QMIX** (eq 7, monotonic mixing): $Q_{tot} = f_s\big(Q_1,\dots,Q_n\big),\quad \frac{\partial Q_{tot}}{\partial Q_i} \geq 0\ \forall i$, with $f_s$ a hypernetwork conditioned on the global state $s$. Monotonicity guarantees IGM but is a **lossy** decomposition (cannot represent non-monotonic joint values — the pincer counterexample in README §7.2).

In [ ]:
# SDK-only entry: build the SDK and read the figure paths from config.
from pathlib import Path

from src.sdk.sdk import MarlSDK
from src.utils.config_loader import load_config

cfg = load_config()
sdk = MarlSDK(cfg)
fig_dir = Path(cfg["paths"]["figures_dir"])
print("algorithms compared:", ["qmix", "vdn", "iql"])
print("figures dir:", fig_dir)

In [ ]:
# Display F1-F6 (regenerate first: scripts/run_results.py -> src.results.make_figures).
from IPython.display import Image, display

figures = [
    "learning_curves",       # F1 — capture rate vs round (mean +/- SE)
    "loss_curves",           # F2 — training loss
    "baseline_comparison",   # F5 — IQL vs VDN vs QMIX
    "scaling",               # F6 — capture rate vs grid size
    "sensitivity_view_radius",  # §9 sensitivity sweep
]
for name in figures:
    path = fig_dir / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"[pending] {path} — run the training matrix then make_figures")

## Bibliography

[1] Oliehoek & Amato, *A Concise Introduction to Dec-POMDPs* (2016). [2] Sunehag et al., *Value-Decomposition Networks* (VDN), arXiv:1706.05296. [3] Rashid et al., *QMIX*, arXiv:1803.11485. [4] Lowe et al., *MADDPG*, arXiv:1706.02275. [5] Lin et al., pursuit-evasion curriculum. [6] Foerster et al., *COMA*, arXiv:1705.08926. [7] OLoRA / orthonormal-LoRA reference (§III). [8] Yu et al., *MAPPO*, arXiv:2103.01955. [9] *Weighted QMIX*, arXiv:2006.10800. [10] *QPLEX*, arXiv:2008.01062. [11] Tan, *Independent Q-Learning* (1993).